# Iteration 4 — Full Fine-Tune + QLoRA r=128 (Overnight Run)

Two experiments in sequence:
- **Experiment A: Full fine-tune** — all 1.5B parameters trainable, 3 epochs, LR 2e-5
- **Experiment B: QLoRA r=128** — 4-bit quantized, rank 128 (4x previous), 8 epochs, LR 5e-5

Both train on all high+medium persona strength examples (~550 examples).

**Full fine-tune runs first** so you can verify it starts correctly before sleeping.

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf /content/PET_QLORA_POET
!git clone https://github.com/thejesusyung/melancholic-poet-qlora.git /content/PET_QLORA_POET

In [ ]:
%cd /content/PET_QLORA_POET/melancholic_poet_qlora
!pip install -r requirements.txt -q

import os, sys
os.environ["PYTHONPATH"] = "/content/PET_QLORA_POET/melancholic_poet_qlora/src"
sys.path.insert(0, "/content/PET_QLORA_POET/melancholic_poet_qlora/src")

## 2. Prepare training data (high + medium persona strength)

In [ ]:
!python scripts/prepare_root_data.py \
  --source_dir /content/PET_QLORA_POET/data \
  --min_persona_strength medium

## 3. Experiment A: Full Fine-Tune (runs first — verify it starts before sleeping!)

All 1.5B parameters trainable, 3 epochs, LR 2e-5. This will take ~2-4 hours on T4.

In [ ]:
!PYTHONPATH=/content/PET_QLORA_POET/melancholic_poet_qlora/src python train.py --config configs/full_finetune_qwen25_15b.yaml

## 4. Experiment B: QLoRA r=128 (runs after full fine-tune)

4-bit quantized, LoRA rank 128 (4x previous), 8 epochs.

In [ ]:
!PYTHONPATH=/content/PET_QLORA_POET/melancholic_poet_qlora/src python train.py --config configs/qlora_r128_qwen25_15b.yaml

## 5. Upload results to S3

- **Full model** → `s3://bucket/full_model/`
- **QLoRA r=128 adapter** → `s3://bucket/adapters/persona_only/`

In [ ]:
!pip install awscli -q

import os
os.environ["AWS_ACCESS_KEY_ID"] = ""       # fill in
os.environ["AWS_SECRET_ACCESS_KEY"] = ""   # fill in
os.environ["AWS_DEFAULT_REGION"] = "us-east-2"

In [ ]:
S3_BUCKET = "melancholic-poet-qlora-901059153423-us-east-2-an"

# Full fine-tuned model (~3GB)
!aws s3 sync outputs/full_finetune_qwen25_15b/full_model/ s3://{S3_BUCKET}/full_model/ --delete

# QLoRA r=128 adapter → persona_only slot
!aws s3 sync outputs/qlora_r128_qwen25_15b/adapter/ s3://{S3_BUCKET}/adapters/persona_only/ --delete

print("Done. Restart the EC2 Gradio app to load the new models.")